# Actividad 5: Método de Uniformización

Se presenta un *notebook* con los ejercicios para programar de esta actividad.

## Ejercicio 3

**Teorema (matriz $\mathbf{P(t)}$)**: La matriz de probabilidad de transición $P(t) = [p_{i,j} (t)]$ está dada por

$$
P(t) = \sum_{k=0}^{\infty}e^{−rt}\frac{(rt)^k}{k!}\hat{P}^k
$$

(Ejercicio para programar) Este último teorema permite aproximar P(t) usando los primeros M términos de la serie infinita. Se obtienen buenos resultados si se elije

$$
M \approx max\{rt + 5\sqrt{rt} ,~ 20\}
$$

1. Use esta propuesta para calcular $P(0.5), P(1)\text{ y }P(5)$ para la matriz $R$ del ejercicio 1.
2. ¿Se verifica la ecuación de Chapman-Kolmogorov $P(1) = P(0.5)P(0.5)$

Matriz $R$ del ejercicio 1: (r=6)

$$
\begin{pmatrix}
    0 & 2 & 3 & 0 \\
    4 & 0 & 2 & 0 \\
    0 & 2 & 0 & 2 \\
    1 & 0 & 3 & 0
\end{pmatrix}
$$

In [21]:
import numpy as np
from scipy.special import factorial
import sympy as sp

sp.init_printing()

# Matriz de tasas R
R_np = np.array([
    [0, 2, 3, 0],
    [4, 0, 2, 0],
    [0, 2, 0, 2],
    [1, 0, 3, 0]
], dtype=float)

r = 6.0
N = len(R_np)

# --- CONSTRUCCIÓN DE \hat{P} por definición ---
P_hat = np.zeros((N, N))
r_lista = R_np.sum(axis=1) 

for i in range(N):
    for j in range(N):
        if i == j:
            P_hat[i, j] = 1.0 - (r_lista[i] / r)
        else:
            P_hat[i, j] = R_np[i, j] / r

# Mostrar la matriz \hat{P}
print("=== MATRIZ P_hat (Uniformizada) ===")
P_hat_sp = sp.Matrix(np.round(P_hat, 5))
display(P_hat_sp)
print("\n" + "="*60 + "\n")


def calcular_P_t(t, r, P_hat): # Aproximación con sumas parciales
    M = int(np.ceil(max(r * t + 5 * np.sqrt(r * t), 20))) # Tomar el ceiling del máximo para que M sea entero
    P_t = np.zeros((N, N))
    P_hat_k = np.eye(N) # guardar las P_hat^k
    
    for k in range(M + 1):
        termino_poisson = np.exp(-r * t) * ((r * t) ** k) / factorial(k)
        P_t += termino_poisson * P_hat_k
        P_hat_k = P_hat_k @ P_hat
        
    return P_t, M

# --- PUNTO 1: Calcular P(t) para t = 0.5, 1, 5 ---
tiempos = [0.5, 1.0, 5.0]
resultados_P = {}


print("=== PUNTO 1: Cálculo de P(t) ===")
for t in tiempos:
    P_t, M = calcular_P_t(t, r, P_hat)
    resultados_P[t] = [M,P_t] # Guardar en un diccionario
    
    # Mostrar matriz
    P_t_sp = sp.Matrix(np.round(P_t, 5))
    print(f"\nMatriz P({t}):, con M={M}")
    display(P_t_sp)

print("\n" + "="*60 + "\n")

# --- PUNTO 2: Verificación de Chapman-Kolmogorov ---
print("=== PUNTO 2: Verificación de Chapman-Kolmogorov ===")
P_05 = resultados_P[0.5][1]
P_1 = resultados_P[1.0][1]

# Multiplicar P(0.5) * P(0.5)
P_1_combinado = P_05 @ P_05

# Convertir a SymPy para la presentación visual
P_1 = sp.Matrix(np.round(P_1, 5))
P_1_combinado = sp.Matrix(np.round(P_1_combinado, 5))

print("\n"+"="*60)
print("\n¿Se verifica la ecuación P(1) = P(0.5)P(0.5)?")
print("\nMatriz P(1) calculada directo de la serie:")
display(P_1)

print("\nMatriz producto P(0.5) * P(0.5):")
display(P_1_combinado)

print("Conclusión: Sí se verifica la ecuación de Chapman-Kolmogorov")

=== MATRIZ P_hat (Uniformizada) ===


⎡0.16667  0.33333    0.5      0.0  ⎤
⎢                                  ⎥
⎢0.66667    0.0    0.33333    0.0  ⎥
⎢                                  ⎥
⎢  0.0    0.33333  0.33333  0.33333⎥
⎢                                  ⎥
⎣0.16667    0.0      0.5    0.33333⎦



=== PUNTO 1: Cálculo de P(t) ===

Matriz P(0.5):, con M=20


⎡0.25061  0.21696  0.38666  0.14577⎤
⎢                                  ⎥
⎢0.25313  0.23836  0.37441  0.13409⎥
⎢                                  ⎥
⎢0.16912  0.19361  0.4203   0.21696⎥
⎢                                  ⎥
⎣0.15802  0.15744  0.39833  0.28621⎦


Matriz P(1.0):, con M=20


...



=== PUNTO 2: Verificación de Chapman-Kolmogorov ===


¿Se verifica la ecuación P(1) = P(0.5)P(0.5)?

Matriz P(1) calculada directo de la serie:


⎡0.20615  0.2039   0.39871  0.19124⎤
⎢                                  ⎥
⎢0.20828  0.20534  0.3979   0.18847⎥
⎢                                  ⎥
⎢0.19676  0.19838  0.40096  0.2039 ⎥
⎢                                  ⎥
⎣0.19205   0.194   0.40147  0.21248⎦


Matriz producto P(0.5) * P(0.5):


⎡0.20615  0.2039   0.39871  0.19124⎤
⎢                                  ⎥
⎢0.20828  0.20534  0.3979   0.18847⎥
⎢                                  ⎥
⎢0.19676  0.19838  0.40096  0.2039 ⎥
⎢                                  ⎥
⎣0.19205   0.194   0.40147  0.21248⎦

Conclusión: Sí se verifica la ecuación de Chapman-Kolmogorov


## Ejercicio 4: 

**Teorema (Cotas de error para $\mathbf{P(t)}$)**: Para un $t \geq 0$ fijo, sea

$$
P^M(t) = [P_{i,j}^M(t)] = \sum_{k=0}^M e^{-rt} \frac{(rt)^k}{k!}\hat{P}^k
$$

Entonces 

$$
|p_{i,j}(t)-p_{i,j}^M (t)| \leq \sum_{k=M+1}^{\infty} e^{-rt} \frac{(rt)^k}{k!}~~~\text{ para todo } 1\leq i,j \leq N
$$

(Ejercicio para programar) Este teorema se puede usar así: Suponga que se desea calcular $P(t)$ con una tolerancia $\varepsilon$. Elija $M$ tal que

$$
\sum_{k=M+1}^{\infty} e^{-rt} \frac{(rt)^k}{k!} \leq \varepsilon
$$

Y se puede implementar de acuerdo al siguiente algoritmo de uniformización para $P(t)$:
1. Dados $R, t, 0 < ε < 1$
2. Calcular $r$ usando la igualdad en la definición ($r=max\{r_i\}$).
3. Calcular $\hat{P}$.
4. $A = \hat{P};~ B = e^{−rt}I;~ c = e^{−rt};~ sum = c; k = 1$
5. Mientras $sum < 1 − \varepsilon$ hacer:
$$ \displaystyle
c = c ∗ \frac{rt}{k} $$
$$B = B + cA $$
$$A = A\hat{P}$$ 
$$sum = sum + c$$ 
$$k = k + 1 $$
6. B está a $\varepsilon$ de P(t).

Repita el ejercicio 3 aplicando este algoritmo con una tolerancia $\varepsilon = 0.00001$ (indique el valor correspondiente de $M$ en cada caso). Compare los resultados.

In [18]:
import numpy as np
import sympy as sp

# Inicializar el entorno de impresión nativo de SymPy
sp.init_printing()

# Paso 1. Dados R, t, varepsilon
R = np.array([
    [0, 2, 3, 0],
    [4, 0, 2, 0],
    [0, 2, 0, 2],
    [1, 0, 3, 0]
], dtype=float)

varepsilon = 0.00001
N = len(R)

# Paso 2: Calcular r usando la igualdad de la definición (r = máx {r_i})
r_lista = R.sum(axis=1) 
r = float(np.max(r_lista))

# Paso 3: Calcular \hat{P} 
P_hat = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        if i == j:
            P_hat[i, j] = 1.0 - (r_lista[i] / r)
        else:
            P_hat[i, j] = R[i, j] / r

print("="*15 + " PARÁMETROS INICIALES " + "="*15)
print(f"Tasa r (máximo de las filas): {r}")
print("Matriz P_hat:")
display(sp.Matrix(np.round(P_hat, 5)).applyfunc(sp.sympify))
print("\n" + "="*60 + "\n")


def algoritmo_uniformizacion(t, r, P_hat, eps):
    # Paso 4: Inicialización de variables tal como pide el pseudocódigo
    A = P_hat.copy()
    B = np.exp(-r * t) * np.eye(N)
    c = np.exp(-r * t)
    suma = c
    k = 1
    
    # Paso 5: Mientras sum < 1 - varepsilon hacer
    while suma < (1.0 - eps):
        c = c * (r * t) / k
        B = B + c * A
        A = A @ P_hat
        suma = suma + c
        k = k + 1

    # El enunciado dice que k = M + 1 ==> M = k-1 
    M_final = k - 1
    return B, M_final

# --- EVALUACIÓN PARA LOS TIEMPOS DEL EJERCICIO 3 (t = 0.5, 1, 5) ---
tiempos = [0.5, 1.0, 5.0]
resultados_P = {}

print("=== REPETICIÓN DEL EJERCICIO 3 CON ALGORITMO DINÁMICO ===")
for t in tiempos:
    P_t_np, M_calculado = algoritmo_uniformizacion_dinamico(t, r, P_hat, varepsilon)
    resultados_P[t] = P_t_np
    
    P_t_sp = sp.Matrix(np.round(P_t_np, 5)).applyfunc(sp.sympify)
    print(f"\nPara t = {t} -> Valor de M requerido para tolerancia: {M_calculado}")
    display(P_t_sp)

print("\n" + "="*60 + "\n")

# --- COMPARACIÓN DE CHAPMAN-KOLMOGOROV BAJO EL NUEVO ALGORITMO ---
print("=== COMPROBACIÓN DE CHAPMAN-KOLMOGOROV ===")
P_05 = resultados_P[0.5]
P_1_directo = resultados_P[1.0]

# Multiplicar P(0.5) * P(0.5)
P_1_combinado = P_05 @ P_05

P_1_directo_sp = sp.Matrix(np.round(P_1_directo, 5)).applyfunc(sp.sympify)
P_1_combinado_sp = sp.Matrix(np.round(P_1_combinado, 5)).applyfunc(sp.sympify)

print("\nMatriz P(1) calculada directo con el algoritmo dinámico:")
display(P_1_directo_sp)

print("\nMatriz producto P(0.5) * P(0.5):")
display(P_1_combinado_sp)

verificacion = np.allclose(P_1_directo, P_1_combinado, atol=1e-5)
print(f"\n¿Se verifica la ecuación P(1) = P(0.5)P(0.5)? -> {verificacion}")

=============== PARÁMETROS INICIALES ===============
Tasa r (máximo de las filas): 6.0
Matriz P_hat:


⎡0.16667  0.33333    0.5      0.0  ⎤
⎢                                  ⎥
⎢0.66667    0.0    0.33333    0.0  ⎥
⎢                                  ⎥
⎢  0.0    0.33333  0.33333  0.33333⎥
⎢                                  ⎥
⎣0.16667    0.0      0.5    0.33333⎦



=== REPETICIÓN DEL EJERCICIO 3 CON ALGORITMO DINÁMICO ===

Para t = 0.5 -> Valor de M requerido para tolerancia: 13


⎡0.25061  0.21696  0.38666  0.14577⎤
⎢                                  ⎥
⎢0.25313  0.23836  0.37441  0.13409⎥
⎢                                  ⎥
⎢0.16912  0.19361  0.4203   0.21696⎥
⎢                                  ⎥
⎣0.15802  0.15744  0.39833  0.28621⎦


Para t = 1.0 -> Valor de M requerido para tolerancia: 19


...



=== COMPROBACIÓN DE CHAPMAN-KOLMOGOROV ===

Matriz P(1) calculada directo con el algoritmo dinámico:


⎡0.20615  0.2039   0.39871  0.19124⎤
⎢                                  ⎥
⎢0.20828  0.20534  0.3979   0.18847⎥
⎢                                  ⎥
⎢0.19676  0.19838  0.40096  0.2039 ⎥
⎢                                  ⎥
⎣0.19205   0.194   0.40147  0.21248⎦


Matriz producto P(0.5) * P(0.5):


⎡0.20615  0.2039   0.39871  0.19123⎤
⎢                                  ⎥
⎢0.20828  0.20534  0.3979   0.18847⎥
⎢                                  ⎥
⎢0.19676  0.19838  0.40096  0.2039 ⎥
⎢                                  ⎥
⎣0.19205   0.194   0.40147  0.21248⎦


¿Se verifica la ecuación P(1) = P(0.5)P(0.5)? -> True


## Conclusión:

Al comparar los resultados obtenidos con este algoritmo contra los obtenidos contra el anterior, es fácil notar que las matrices resultantes son escencialmente iguales, con posibles diferencias cumputacionales o de redondeo mínimas.

Pero además, existe una diferencia: El valor de $M$, pues en el primer algoritmo no era posible obtener un $M<20$, y aquí sí. Esto es evidencia para decir que este algoritmo es mejor que el anterior pues se obtienen los mismos resultados con menor cantidad de cálculos.
En el caso de $t=5$, donde en el algoritmo inicial se obtuvo un valor $M=58$, con este segundo algoritmo se obtiene el mismo resultado con $M=56$, comprobando una menor cantidad de iteraciones incluso en casos más altos.